# End-to-End Intent Parser PoC — Stages 1, 2 & 3 Unified

**Source notebooks:**
- Stage 1 & 2: `intent_parsing_classification.ipynb`
- Stage 3: `hybrid_item_validation_poc.ipynb`

**Full pipeline:**

```
Raw NL Query
     │
     ▼
┌──────────────────────────────────┐
│  STAGE 1 — IntentClassifier      │  Ollama qwen3:8b (instructor)
│  ParsedIntent {intent, confidence}│
└─────────────┬────────────────────┘
              │ gate: _PROCUREMENT_INTENTS
              ▼
┌──────────────────────────────────┐
│  STAGE 2 — BecknIntentParser     │  Ollama qwen3:8b / qwen3:1.7b
│  BecknIntent {item, descriptions, │  (heuristic complexity router)
│  quantity, location, budget, ...} │
└─────────────┬────────────────────┘
              │ item + descriptions
              ▼
┌──────────────────────────────────┐
│  STAGE 3 — Hybrid BPP Validation │  all-MiniLM-L6-v2 (384d, local CPU)
│                                   │  + pgvector HNSW ANN search
│  ┌────────┬──────────┬─────────┐  │
│  │≥ 0.85  │0.45–0.84 │ < 0.45  │  │
│  │VALIDATED│AMBIGUOUS │CACHE    │  │
│  │        │suggestions│MISS    │  │
│  │        │          │  │      │  │
│  │        │          │  ▼ MCP  │  │
│  │        │          │  fallback│  │
│  │        │          │  Path B │  │
│  └────────┴──────────┴─────────┘  │
└──────────────────────────────────┘
              │
              ▼
  ParseResponse {intent, confidence,
   beckn_intent, routed_to, validation}
```

## 0. Configuration

**Primary stack (always active):**
- **Ollama** → instructor-wrapped client for Stage 1 & 2 LLM extraction
- **sentence-transformers** → `all-MiniLM-L6-v2` (384d) for Stage 3 embeddings (local CPU)
- **PostgreSQL** + pgvector → `vector(384)` HNSW semantic cache

**Cloud fallback (last resort only — Local-First principle):**
- **Anthropic Claude** (`claude-sonnet-4-6`) → complex Query Broadening and Senior Auditor.
  Activated only when `ANTHROPIC_API_KEY` is set in the environment.
  Gracefully disabled if the key is absent — the pipeline logs a placeholder instead.

Threshold calibration for all-MiniLM-L6-v2 (empirical):

| Zone | Threshold | Meaning |
|---|---|---|
| VALIDATED | ≥ 0.85 | Near-identical match in cache |
| AMBIGUOUS | ≥ 0.45 | Loosely related — surface suggestions |
| CACHE_MISS | < 0.45 | Unknown item — trigger MCP fallback → recovery |

In [1]:
import os

# ── Stage 1 & 2 — Ollama LLM ──────────────────────────────────────────────
OLLAMA_BASE_URL   = "http://localhost:11434/v1"
OLLAMA_API_KEY    = "qwen3"          # Ollama ignores this; OpenAI client requires it
COMPLEX_MODEL     = "qwen3:8b"       # Stage 2 complex queries
SIMPLE_MODEL      = "qwen3:1.7b"     # Stage 2 simple queries
STAGE1_MODEL      = "qwen3:8b"       # Stage 1 always uses this

# ── Stage 3 — sentence-transformers all-MiniLM-L6-v2 (384d, local CPU) ───
# normalize_embeddings=True → unit vector → dot product = cosine similarity
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
EMBEDDING_DIM   = 384

# ── PostgreSQL (peer auth) ─────────────────────────────────────────────────
DB_CONFIG = {
    "dbname": "procurement_agent",
    "user":   "carbaje",
}

# ── Stage 3 thresholds — calibrated for all-MiniLM-L6-v2 cosine similarity ─
HNSW_EF_SEARCH      = 100
VALIDATED_THRESHOLD = 0.85
AMBIGUOUS_THRESHOLD = 0.45

# ── Cloud Fallback — Anthropic Claude (last resort only) ───────────────────
# Local-First principle: Claude is invoked ONLY on confirmed failure paths.
# Set ANTHROPIC_API_KEY in the shell before running to enable cloud calls.
# If the key is absent, call_claude_fallback() returns a graceful placeholder.
CLAUDE_MODEL            = "claude-sonnet-4-6"
ANTHROPIC_API_KEY       = os.environ.get("ANTHROPIC_API_KEY", "")
CLAUDE_FALLBACK_ENABLED = bool(ANTHROPIC_API_KEY)

print(f"Ollama base URL       : {OLLAMA_BASE_URL}")
print(f"LLM models            : {COMPLEX_MODEL} / {SIMPLE_MODEL}")
print(f"Embedding model       : {EMBEDDING_MODEL} ({EMBEDDING_DIM}d, local CPU)")
print(f"DB target             : {DB_CONFIG['dbname']}")
print(f"Claude fallback       : {'ENABLED' if CLAUDE_FALLBACK_ENABLED else 'DISABLED (no ANTHROPIC_API_KEY)'}")

Ollama base URL       : http://localhost:11434/v1
LLM models            : qwen3:8b / qwen3:1.7b
Embedding model       : all-MiniLM-L6-v2 (384d, local CPU)
DB target             : procurement_agent
Claude fallback       : DISABLED (no ANTHROPIC_API_KEY)


## 1. Dependencies

In [2]:
from __future__ import annotations

import os
import re
import numpy as np
import psycopg2
import psycopg2.extras
import instructor
import anthropic

from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional

from openai import OpenAI
from pgvector.psycopg2 import register_vector
from pydantic import BaseModel, Field, field_validator
from sentence_transformers import SentenceTransformer

print(f"instructor : {instructor.__version__}")
print(f"anthropic  : {anthropic.__version__}")
print("All dependencies loaded.")

/home/carbaje/anaconda3/envs/infosys_project/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


instructor : 1.15.1
anthropic  : 0.97.0
All dependencies loaded.


## 2. LLM Clients

**Local clients (always initialized):**
- `_llm_client` — Ollama instructor client for Stage 1 & 2
- `_st_model` / `_get_st_model()` — SentenceTransformer singleton for Stage 3 embeddings

**Cloud client (initialized only if `ANTHROPIC_API_KEY` is set):**
- `_anthropic_client` — Anthropic SDK client for Claude fallback

| Variable | Purpose | Backend |
|---|---|---|
| `_llm_client` | Stage 1 & 2 structured extraction via instructor | Ollama qwen3:8b / qwen3:1.7b |
| `_st_model` / `_get_st_model()` | Stage 3 embeddings (384d) | all-MiniLM-L6-v2, local CPU |
| `_anthropic_client` | Query Broadening + Senior Auditor (last resort) | Anthropic claude-sonnet-4-6 |

In [3]:
# ── Stage 1 & 2: Ollama via instructor ───────────────────────────────────
# instructor.Mode.JSON is required for Ollama
_ollama_raw_client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=OLLAMA_API_KEY,
)
_llm_client = instructor.from_openai(_ollama_raw_client, mode=instructor.Mode.JSON)
print(f"Ollama instructor client ready (mode=JSON, base_url={OLLAMA_BASE_URL})")

# ── Stage 3: sentence-transformers (lazy singleton) ────────────────────────
_st_model: Optional[SentenceTransformer] = None


def _get_st_model() -> SentenceTransformer:
    """Lazy singleton for the sentence-transformers embedding model."""
    global _st_model
    if _st_model is None:
        _st_model = SentenceTransformer(EMBEDDING_MODEL)
    return _st_model


print(f"SentenceTransformer lazy factory ready (model={EMBEDDING_MODEL}, {EMBEDDING_DIM}d)")

# ── Cloud Fallback: Anthropic Claude (last resort, disabled if no key) ────
_anthropic_client: Optional[anthropic.Anthropic] = (
    anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    if CLAUDE_FALLBACK_ENABLED
    else None
)
if CLAUDE_FALLBACK_ENABLED:
    print(f"Anthropic client ready (model={CLAUDE_MODEL})")
else:
    print("Anthropic client DISABLED — set ANTHROPIC_API_KEY to enable Claude fallback")

print("Stack: Local-First — cloud calls only on confirmed failure paths.")

Ollama instructor client ready (mode=JSON, base_url=http://localhost:11434/v1)
SentenceTransformer lazy factory ready (model=all-MiniLM-L6-v2, 384d)
Anthropic client DISABLED — set ANTHROPIC_API_KEY to enable Claude fallback
Stack: Local-First — cloud calls only on confirmed failure paths.


## 3. Stage 3 — Database Schema (vector(384))

The `bpp_catalog_semantic_cache` table must use `vector(384)` to match `all-MiniLM-L6-v2` output.
The cell below drops and recreates the table (and its HNSW index) with the correct dimension.
This is idempotent — safe to re-run; the cache is ephemeral PoC data.

In [4]:
# Recreate bpp_catalog_semantic_cache with vector(384)
# all-MiniLM-L6-v2 outputs 384d — inserting a 384d vector into a 1536d or 768d column
# causes a pgvector dimension mismatch error, so the table is recreated here.
RECREATE_SCHEMA_SQL = """
    DROP TABLE IF EXISTS bpp_catalog_semantic_cache CASCADE;

    CREATE TABLE bpp_catalog_semantic_cache (
        id                  UUID            PRIMARY KEY DEFAULT gen_random_uuid(),
        item_name           TEXT            NOT NULL,
        item_embedding      vector(384)     NOT NULL,
        descriptions        TEXT[],
        bpp_id              TEXT            NOT NULL,
        bpp_uri             TEXT            NOT NULL,
        provider_id         TEXT,
        category_tag        TEXT,
        source              TEXT            NOT NULL
                                CHECK (source IN ('bpp_publish', 'mcp_feedback')),
        embedding_strategy  TEXT            NOT NULL
                                CHECK (embedding_strategy IN ('item_name_only', 'item_name_and_specs')),
        created_at          TIMESTAMPTZ     NOT NULL DEFAULT NOW(),
        last_seen_at        TIMESTAMPTZ     NOT NULL DEFAULT NOW(),
        hit_count           INTEGER         NOT NULL DEFAULT 0,
        CONSTRAINT uq_bpp_catalog_item_per_bpp UNIQUE (item_name, bpp_id)
    );

    CREATE INDEX hnsw_bpp_catalog_embedding_cosine
        ON bpp_catalog_semantic_cache
        USING hnsw (item_embedding vector_cosine_ops)
        WITH (m = 16, ef_construction = 64);
"""


def get_db_connection() -> psycopg2.extensions.connection:
    """Open a psycopg2 connection with pgvector type adapters registered."""
    conn = psycopg2.connect(**DB_CONFIG)
    register_vector(conn)
    return conn


# Apply schema migration
with get_db_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(RECREATE_SCHEMA_SQL)
    conn.commit()
print(f"Schema OK — bpp_catalog_semantic_cache recreated with vector({EMBEDDING_DIM}) + HNSW index.")

# Smoke test
with get_db_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM bpp_catalog_semantic_cache;")
        current_row_count = cur.fetchone()[0]
print(f"DB connection OK — procurement_agent reachable. Rows: {current_row_count}")

Schema OK — bpp_catalog_semantic_cache recreated with vector(384) + HNSW index.
DB connection OK — procurement_agent reachable. Rows: 0


## 4. Stage 1 & 2 — Pydantic Schemas

These are the shared output contracts imported verbatim from `intent_parsing_classification.ipynb`.

| Schema | Stage | Role |
|---|---|---|
| `ParsedIntent` | 1 | Intent classification result |
| `BecknIntent` | 2 | Fully structured Beckn discover payload |

In [5]:
class ParsedIntent(BaseModel):
    """Stage 1 output — intent classification result."""

    intent: str = Field(
        description=(
            "Primary detected intent. Use PascalCase and be specific, "
            "e.g. 'SearchProduct', 'RequestQuote', 'TrackOrder'. "
            "If it does not fit any known category, invent a descriptive name."
        )
    )
    product_name: Optional[str] = Field(
        default=None,
        description="Name of the product mentioned, if any. None if not applicable."
    )
    quantity: Optional[int] = Field(
        default=None,
        description="Requested quantity, if explicitly mentioned."
    )
    confidence: float = Field(
        description="Confidence level in the classification, between 0.0 and 1.0."
    )
    reasoning: str = Field(
        description="Brief explanation of why that intent was chosen."
    )

    @field_validator("confidence")
    @classmethod
    def confidence_in_range(cls, v: float) -> float:
        if not 0.0 <= v <= 1.0:
            raise ValueError(f"confidence must be between 0 and 1, got: {v}")
        return round(v, 2)


print("ParsedIntent schema defined.")

ParsedIntent schema defined.


In [6]:
_CITY_COORDINATES: dict[str, str] = {
    "bangalore":  "12.9716,77.5946",
    "bengaluru":  "12.9716,77.5946",
    "mumbai":     "19.0760,72.8777",
    "delhi":      "28.7041,77.1025",
    "new delhi":  "28.6139,77.2090",
    "chennai":    "13.0827,80.2707",
    "hyderabad":  "17.3850,78.4867",
    "pune":       "18.5204,73.8567",
    "kolkata":    "22.5726,88.3639",
}


def resolve_location(raw_location_text: str) -> str:
    """Deterministic lookup: known city name → lat,lon. Unknown text passes through."""
    normalized = raw_location_text.strip().lower()
    for city_name, coordinates in _CITY_COORDINATES.items():
        if city_name in normalized:
            return coordinates
    return raw_location_text


_DELIVERY_KEYWORDS = frozenset({
    "delivery", "deliver", "timeline", "deadline",
    "days", "weeks", "hours", "within",
})
_BUDGET_KEYWORDS = frozenset({
    "budget", "price", "cost", "rupee", "rupees",
    "inr", "usd", "per unit", "per sheet", "per meter",
    "under", "maximum", "max",
})


def is_complex_request(query: str) -> bool:
    """Heuristic router — returns True for queries that need the larger model."""
    if len(query) > 120:
        return True
    if len(re.findall(r"\b\d+(?:\.\d+)?\b", query)) >= 2:
        return True
    lower = query.lower()
    if any(kw in lower for kw in _DELIVERY_KEYWORDS):
        return True
    if any(kw in lower for kw in _BUDGET_KEYWORDS):
        return True
    return False


class BudgetConstraints(BaseModel):
    max: float
    min: float = 0.0


class BecknIntent(BaseModel):
    """Stage 2 output — fully structured Beckn discover payload."""

    item: str = Field(description="Product name")
    descriptions: list[str] = Field(
        description="Technical specs extracted from the query, e.g. ['80gsm', 'A4', 'Cat6']"
    )
    quantity: int = Field(description="Quantity requested")
    location_coordinates: str = Field(
        description=(
            "Location as 'lat,lon'. Resolve known Indian cities to their coordinates "
            "using the provided lookup. Return raw text if unknown."
        )
    )
    delivery_timeline: int = Field(
        description="Delivery time in hours. Convert: 1 day = 24h, 1 week = 168h."
    )
    budget_constraints: BudgetConstraints = Field(
        description="Budget limits. If only an upper bound is given, set max to that and min to 0."
    )

    @field_validator("delivery_timeline")
    @classmethod
    def timeline_positive(cls, v: int) -> int:
        if v <= 0:
            raise ValueError(f"delivery_timeline must be > 0, got {v}")
        return v

    @field_validator("location_coordinates")
    @classmethod
    def apply_location_lookup(cls, v: str) -> str:
        return resolve_location(v)


print("BecknIntent + helpers defined.")

BecknIntent + helpers defined.


## 5. Stage 1 — `IntentClassifier`

In [7]:
_STAGE1_SYSTEM_PROMPT = """
You are an intent classifier for a procurement system based on the Beckn protocol.
Analyze the user's query and return the most precise intent.

Domain context:
- Users search for industrial products, equipment, and materials
- They may request quotes (RFQ), place orders, or track purchases
- Suppliers are on a decentralized Beckn network
""".strip()


class IntentClassifier:
    """Stage 1: classifies a raw query into a ParsedIntent."""

    def __init__(self, model: str = STAGE1_MODEL, max_retries: int = 3):
        self.model       = model
        self.max_retries = max_retries

    def classify(self, query: str) -> ParsedIntent:
        return _llm_client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": _STAGE1_SYSTEM_PROMPT},
                {"role": "user",   "content": query},
            ],
            response_model=ParsedIntent,
            max_retries=self.max_retries,
        )


print("IntentClassifier defined.")

IntentClassifier defined.


## 6. Stage 2 — `BecknIntentParser`

In [8]:
_STAGE2_SYSTEM_PROMPT = """
You are a procurement data extractor for the Beckn protocol.
Extract structured data from the user query.

Rules:
- Extract all technical specs into descriptions (e.g. "80gsm", "A4", "Cat6", "2 inch")
- Convert all time expressions to hours: 1 day = 24h, 1 week = 168h
- Use only numeric values for budget fields (no currency symbols)
- Resolve locations using this lookup:
  Bangalore/Bengaluru=12.9716,77.5946 | Mumbai=19.0760,72.8777 | Delhi=28.7041,77.1025
  Chennai=13.0827,80.2707 | Hyderabad=17.3850,78.4867 | Pune=18.5204,73.8567
  Kolkata=22.5726,88.3639
- If location is not in the list, return the raw location text
- If only an upper budget is mentioned, set min to 0
""".strip()


class BecknIntentParser:
    """Stage 2: extracts a BecknIntent from a procurement query using complexity routing."""

    def __init__(
        self,
        complex_model: str = COMPLEX_MODEL,
        simple_model:  str = SIMPLE_MODEL,
    ):
        self.complex_model    = complex_model
        self.simple_model     = simple_model
        self.last_model_used  = complex_model

    def _select_model(self, query: str) -> str:
        return self.complex_model if is_complex_request(query) else self.simple_model

    def parse(self, query: str) -> BecknIntent:
        selected_model = self._select_model(query)
        messages = [
            {"role": "system", "content": _STAGE2_SYSTEM_PROMPT},
            {"role": "user",   "content": query},
        ]
        try:
            extracted_intent = _llm_client.chat.completions.create(
                model=selected_model,
                messages=messages,
                response_model=BecknIntent,
                max_retries=3,
            )
            self.last_model_used = selected_model
            return extracted_intent
        except Exception:
            if selected_model != self.complex_model:
                # Escalate to the complex model on failure
                fallback_intent = _llm_client.chat.completions.create(
                    model=self.complex_model,
                    messages=messages,
                    response_model=BecknIntent,
                    max_retries=3,
                )
                self.last_model_used = self.complex_model
                return fallback_intent
            raise


print("BecknIntentParser defined.")

BecknIntentParser defined.


## 7. Stage 3 — Domain Types

In [9]:
class ValidationZone(str, Enum):
    VALIDATED  = "VALIDATED"    # cosine similarity >= VALIDATED_THRESHOLD
    AMBIGUOUS  = "AMBIGUOUS"    # AMBIGUOUS_THRESHOLD <= sim < VALIDATED_THRESHOLD
    CACHE_MISS = "CACHE_MISS"   # sim < AMBIGUOUS_THRESHOLD → MCP fallback → recovery


@dataclass
class ValidationResult:
    zone:                ValidationZone
    similarity:          Optional[float] = None
    matched_item_name:   Optional[str]   = None
    matched_bpp_id:      Optional[str]   = None
    suggestions:         list[str]       = field(default_factory=list)
    mcp_validated:       bool            = False
    path_b_written:      bool            = False
    # Day-2 production fields
    not_found:           bool            = False  # True when cache + MCP both returned nothing
    broadened_item_name: Optional[str]   = None   # Set when Query Broadening was applied

    def __str__(self) -> str:
        parts = [f"zone={self.zone.value}"]
        if self.similarity is not None:
            parts.append(f"similarity={self.similarity:.4f}")
        if self.matched_item_name:
            parts.append(f"matched='{self.matched_item_name}'")
        if self.suggestions:
            parts.append(f"suggestions={self.suggestions}")
        if self.mcp_validated:
            parts.append("mcp_validated=True")
        if self.path_b_written:
            parts.append("path_b_written=True")
        if self.not_found:
            parts.append("not_found=True")
        if self.broadened_item_name:
            parts.append(f"broadened='{self.broadened_item_name}'")
        return f"ValidationResult({', '.join(parts)})"


print("ValidationZone and ValidationResult defined.")

ValidationZone and ValidationResult defined.


## 8. Stage 3 — Embedding Utility (all-MiniLM-L6-v2, 384d)

Uses `sentence-transformers` to embed text locally on CPU — no Ollama, no cloud API.
`normalize_embeddings=True` produces unit vectors, making dot product equivalent to
cosine similarity (compatible with pgvector `<=>` cosine-distance operator).

In [10]:
def embed(text: str) -> np.ndarray:
    """
    Embed text using all-MiniLM-L6-v2 via sentence-transformers (local CPU).
    normalize_embeddings=True → unit vector → dot product = cosine similarity.
    Returns a (384,) float32 ndarray.
    """
    st_model    = _get_st_model()
    unit_vector = st_model.encode(text, normalize_embeddings=True)
    return unit_vector.astype(np.float32)


# Shape + dtype validation
test_embedding = embed("stainless steel flanged ball valve")
assert test_embedding.shape == (EMBEDDING_DIM,), (
    f"Expected ({EMBEDDING_DIM},), got {test_embedding.shape}"
)
print(f"embed() OK — shape={test_embedding.shape}, dtype={test_embedding.dtype}")
print(f"Model: {EMBEDDING_MODEL} | Backend: local CPU | 100% local")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1742.00it/s]


embed() OK — shape=(384,), dtype=float32
Model: all-MiniLM-L6-v2 | Backend: local CPU | 100% local


## 9. Stage 3 — Cache Writers

### Path A — `CatalogCacheWriter` (triggered by BPP `on_discover`)

In [11]:
def write_path_a_cache_row(
    item_name:    str,
    bpp_id:       str,
    bpp_uri:      str,
    provider_id:  Optional[str] = None,
    category_tag: Optional[str] = None,
) -> None:
    """
    CatalogCacheWriter — Path A.
    Embedding input: item_name only (weaker, no specs).
    UPSERT: conflict bumps hit_count + last_seen_at; does NOT overwrite descriptions.
    """
    item_name_embedding = embed(item_name)

    upsert_sql = """
        INSERT INTO bpp_catalog_semantic_cache
            (item_name, item_embedding, bpp_id, bpp_uri,
             provider_id, category_tag, source, embedding_strategy)
        VALUES
            (%(item_name)s, %(embedding)s, %(bpp_id)s, %(bpp_uri)s,
             %(provider_id)s, %(category_tag)s, 'bpp_publish', 'item_name_only')
        ON CONFLICT (item_name, bpp_id) DO UPDATE
            SET last_seen_at = NOW(),
                hit_count    = bpp_catalog_semantic_cache.hit_count + 1;
    """
    with get_db_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(upsert_sql, {
                "item_name":    item_name,
                "embedding":    item_name_embedding,
                "bpp_id":       bpp_id,
                "bpp_uri":      bpp_uri,
                "provider_id":  provider_id,
                "category_tag": category_tag,
            })
        conn.commit()


print("write_path_a_cache_row() defined.")

write_path_a_cache_row() defined.


### Path B — `MCPResultAdapter` (triggered after a successful MCP fallback)

In [12]:
def write_path_b_cache_row(
    item_name:    str,
    descriptions: list[str],
    bpp_id:       str,
    bpp_uri:      str,
    provider_id:  Optional[str] = None,
    category_tag: Optional[str] = None,
) -> None:
    """
    MCPResultAdapter — Path B.
    Embedding input: item_name + ' | ' + join(descriptions) (richer, includes buyer specs).
    UPSERT: conflict upgrades the row — re-embeds with specs, overwrites descriptions.
    """
    enriched_embed_input   = item_name + " | " + " ".join(descriptions)
    enriched_item_embedding = embed(enriched_embed_input)

    upsert_sql = """
        INSERT INTO bpp_catalog_semantic_cache
            (item_name, item_embedding, descriptions, bpp_id, bpp_uri,
             provider_id, category_tag, source, embedding_strategy)
        VALUES
            (%(item_name)s, %(embedding)s, %(descriptions)s, %(bpp_id)s, %(bpp_uri)s,
             %(provider_id)s, %(category_tag)s, 'mcp_feedback', 'item_name_and_specs')
        ON CONFLICT (item_name, bpp_id) DO UPDATE
            SET item_embedding     = EXCLUDED.item_embedding,
                descriptions       = EXCLUDED.descriptions,
                source             = 'mcp_feedback',
                embedding_strategy = 'item_name_and_specs',
                last_seen_at       = NOW(),
                hit_count          = bpp_catalog_semantic_cache.hit_count + 1;
    """
    with get_db_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(upsert_sql, {
                "item_name":    item_name,
                "embedding":    enriched_item_embedding,
                "descriptions": descriptions,
                "bpp_id":       bpp_id,
                "bpp_uri":      bpp_uri,
                "provider_id":  provider_id,
                "category_tag": category_tag,
            })
        conn.commit()


print("write_path_b_cache_row() defined.")

write_path_b_cache_row() defined.


## 10. Stage 3 — HNSW Cache Query + Three-Zone Decision

In [13]:
def query_semantic_cache(
    query_embedding: np.ndarray,
    top_k: int = 3,
) -> list[dict]:
    """
    HNSW cosine ANN search against bpp_catalog_semantic_cache.
    ef_search is set per-session in a separate execute call (psycopg2 limitation).
    Returns rows ordered by descending cosine similarity.
    Similarity = 1 − cosine_distance  (pgvector <=> returns distance, not similarity).
    """
    ann_sql = """
        SELECT
            item_name,
            bpp_id,
            bpp_uri,
            provider_id,
            category_tag,
            source,
            embedding_strategy,
            descriptions,
            1 - (item_embedding <=> %(query_vec)s) AS cosine_similarity
        FROM bpp_catalog_semantic_cache
        ORDER BY item_embedding <=> %(query_vec)s
        LIMIT %(top_k)s;
    """
    with get_db_connection() as conn:
        with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute("SET hnsw.ef_search = %s;", (HNSW_EF_SEARCH,))
            cur.execute(ann_sql, {"query_vec": query_embedding, "top_k": top_k})
            return [dict(row) for row in cur.fetchall()]


def apply_three_zone_threshold(
    ann_hits: list[dict],
) -> tuple[ValidationZone, Optional[dict]]:
    """
    Map the top ANN hit to a ValidationZone.
    VALIDATED  ≥ 0.92 | AMBIGUOUS 0.75–0.91 | CACHE_MISS < 0.75 or no hits.
    """
    if not ann_hits:
        return ValidationZone.CACHE_MISS, None

    top_hit    = ann_hits[0]
    top_sim    = float(top_hit["cosine_similarity"])

    if top_sim >= VALIDATED_THRESHOLD:
        return ValidationZone.VALIDATED, top_hit
    if top_sim >= AMBIGUOUS_THRESHOLD:
        return ValidationZone.AMBIGUOUS, top_hit
    return ValidationZone.CACHE_MISS, None


print("query_semantic_cache() and apply_three_zone_threshold() defined.")

query_semantic_cache() and apply_three_zone_threshold() defined.


## 11. Stage 3 — Mock Functions

### 11a. Mock `CatalogNormalizer` — Path A seed (BPP `on_discover`)

In [14]:
_MOCK_BPP_CATALOG: list[dict] = [
    {
        "item_name":    "Stainless Steel Flanged Ball Valve 2 inch",
        "bpp_id":       "bpp_industrial_001",
        "bpp_uri":      "https://bpp.industrial-supplies.in/beckn",
        "provider_id":  "prov_valve_co",
        "category_tag": "valves",
    },
    {
        "item_name":    "Industrial HDPE Pipe 6 inch",
        "bpp_id":       "bpp_industrial_001",
        "bpp_uri":      "https://bpp.industrial-supplies.in/beckn",
        "provider_id":  "prov_pipes_co",
        "category_tag": "pipes",
    },
    {
        "item_name":    "Safety Helmet Class E Hard Hat",
        "bpp_id":       "bpp_ppe_002",
        "bpp_uri":      "https://bpp.ppe-marketplace.in/beckn",
        "provider_id":  "prov_safety_gear",
        "category_tag": "ppe",
    },
]


def mock_catalog_normalizer_path_a() -> list[dict]:
    """Simulates CatalogNormalizer emitting items from a BPP on_discover response."""
    return _MOCK_BPP_CATALOG


print(f"mock_catalog_normalizer_path_a() defined — {len(_MOCK_BPP_CATALOG)} BPP items.")

mock_catalog_normalizer_path_a() defined — 3 BPP items.


### 11b. Mock MCP `search_bpp_catalog` — Path B fallback

In [15]:
# Keyed by substring present in a realistic LLM-extracted item_name.
# Keys are lowercase; matching is substring-based for LLM output variance tolerance.
_MCP_MOCK_RESPONSES: dict[str, dict] = {
    "ss flanged valve": {
        "found":        True,
        "item_name":    "Stainless Steel Flanged Ball Valve 2 inch",
        "descriptions": ["SS316", "2 inch", "flanged", "PN16", "ball valve"],
        "bpp_id":       "bpp_industrial_001",
        "bpp_uri":      "https://bpp.industrial-supplies.in/beckn",
        "provider_id":  "prov_valve_co",
        "category_tag": "valves",
    },
    # TC3 trigger: Stage 2 will extract an item_name containing "cable"
    # for a networking cable procurement query
    "cable": {
        "found":        True,
        "item_name":    "UTP Cat6 Network Cable",
        "descriptions": ["Cat6", "UTP", "unshielded twisted pair", "550MHz", "23AWG"],
        "bpp_id":       "bpp_it_supplies_003",
        "bpp_uri":      "https://bpp.it-procurement.in/beckn",
        "provider_id":  "prov_network_gear",
        "category_tag": "networking",
    },
}


def mock_mcp_search_bpp_catalog(item_name_from_stage2: str) -> dict:
    """
    Simulates the MCP search_bpp_catalog tool.
    Input: item_name extracted by Stage 2 (BecknIntent.item).
    Returns {found: True, item_name, descriptions, bpp_id, ...} on match,
    or {found: False} when no BPP carries the item.
    """
    normalized_item_name = item_name_from_stage2.lower()
    for mcp_key, mcp_response in _MCP_MOCK_RESPONSES.items():
        if mcp_key in normalized_item_name:
            return mcp_response
    return {"found": False}


print(f"mock_mcp_search_bpp_catalog() defined — {len(_MCP_MOCK_RESPONSES)} mock entries.")

mock_mcp_search_bpp_catalog() defined — 2 mock entries.


### 11c. Cloud Fallback — `call_claude_fallback()` (Last Resort)

Wraps the Anthropic SDK with a **graceful disable** pattern: if `ANTHROPIC_API_KEY` is not set,
the function returns a structured placeholder instead of raising — the pipeline flow continues
and the recovery log records that Claude was unavailable.

This function is **never called on the happy path**. It is only reachable from two places:
1. `broaden_procurement_query()` — if the heuristic produces a degenerate (< 2 token) result
2. `parse_procurement_request()` — Step 4 (Senior Auditor) after full recovery failure

In [16]:
def call_claude_fallback(query: str, context: str) -> str:
    """
    Issue a focused, bounded subtask to Claude (claude-sonnet-4-6).

    Local-First guard: if ANTHROPIC_API_KEY is absent, returns a structured
    placeholder so the recovery flow can continue without a hard failure.

    Args:
        query   : The item name or short text to reason about.
        context : System-role instructions for the subtask (kept tight to < 60 words).

    Returns:
        Claude's single-sentence response, or a placeholder if the key is absent.
    """
    if not CLAUDE_FALLBACK_ENABLED or _anthropic_client is None:
        return (
            f"[CLAUDE DISABLED — set ANTHROPIC_API_KEY to enable] "
            f"Placeholder for: '{query}'"
        )

    message = _anthropic_client.messages.create(
        model      = CLAUDE_MODEL,
        max_tokens = 256,
        system     = context,
        messages   = [{"role": "user", "content": query}],
    )
    return message.content[0].text.strip()


print("call_claude_fallback() defined (CLAUDE_FALLBACK_ENABLED=%s)." % CLAUDE_FALLBACK_ENABLED)

call_claude_fallback() defined (CLAUDE_FALLBACK_ENABLED=False).


### 11d. Query Broadening — `broaden_procurement_query()`

Strips highly specific technical constraints from a `BecknIntent` to produce a **category-level**
item name suitable for a broader semantic cache search.

**Heuristic (always runs first):**
Regex patterns remove standards codes (ASTM, ASME, ISO), material grade codes (S32750, CF8M),
pressure/bore ratings (PN40, DN100), and dimensional specs (2 inch, 100mm).
Category anchors — product type (valve, fitting, cable) and material family (stainless steel,
HDPE) — are preserved.

**Claude fallback (only if heuristic is degenerate):**
If the heuristic strips everything and leaves < 2 tokens, `call_claude_fallback()` is invoked
with a tightly scoped spec-degradation prompt. The response replaces the heuristic output.

In [17]:
# Regex patterns to strip spec-level tokens from item_name during broadening.
# Order matters: more specific patterns first to avoid partial matches.
_SPEC_STRIP_PATTERNS: tuple[str, ...] = (
    r"\bASTM\s+[A-Z]\d+\w*\b",           # ASTM A790, ASTM A351
    r"\bASME\s+[A-Z]\d+[\.\d]*\w*\b",    # ASME B16.5
    r"\b(?:ISO|EN|DIN|ANSI|API)\s*[\d]+\w*\b",  # ISO 9001, EN 10216-1
    r"\b(?:UNS\s+)?S\d{4,5}\b",          # S32750, UNS S32750 (UNS designations)
    r"\bSS\s*3\d{2}\b",                   # SS316, SS304
    r"\bCF[0-9][A-Z]\w*\b",               # CF8M, CF3M (cast stainless grades)
    r"\bPN\s*\d+\b",                       # PN16, PN40 (pressure ratings)
    r"\bDN\s*\d+\b",                       # DN80, DN100 (nominal bore)
    r"\b\d+(?:\.\d+)?\s*(?:inch|mm|cm|bar|psi)\b",  # 2 inch, 100mm, 16 bar
    r"\b(?:class|schedule|sch)\s*\w+\b",  # class E, schedule 40
    r"\b\d+\s*(?:AWG|MHz|GHz|Gbps|Mbps)\b",  # 23AWG, 550MHz (telecom specs)
)


def broaden_procurement_query(
    beckn_intent: BecknIntent,
) -> tuple[str, list[str]]:
    """
    Strip technical specifications from BecknIntent and return a category-level search pair.

    Broadening strategy:
      1. Heuristic regex pass strips standard codes, grade codes, and dimensional specs.
      2. Kept descriptions filter removes numeric and standards-body tokens.
      3. If heuristic leaves < 2 tokens, fall back to call_claude_fallback().

    Returns:
        (broadened_item_name: str, kept_descriptions: list[str])
    """
    # ── Step 1: Heuristic strip on item_name ─────────────────────────────
    broadened_item = beckn_intent.item
    for pattern in _SPEC_STRIP_PATTERNS:
        broadened_item = re.sub(pattern, "", broadened_item, flags=re.IGNORECASE)
    broadened_item = " ".join(broadened_item.split())  # collapse whitespace

    # ── Step 2: Filter descriptions — keep category-level terms only ─────
    _spec_desc_pattern = re.compile(
        r"^[\d]"  # starts with digit (dimensional)
        r"|PN\d+|DN\d+"  # pressure/bore ratings
        r"|ASTM|ASME|ISO|DIN|ANSI"  # standards bodies
        r"|SS\s*3\d{2}|S\d{4,5}",  # material grade codes
        re.IGNORECASE,
    )
    _dimensional_re = re.compile(
        r"\d+\.?\d*\s*(?:inch|mm|bar|psi|AWG|MHz)", re.IGNORECASE
    )
    kept_descriptions = [
        d for d in beckn_intent.descriptions
        if not _spec_desc_pattern.search(d) and not _dimensional_re.search(d)
    ]

    # ── Step 3: Claude fallback if heuristic is degenerate ───────────────
    if len(broadened_item.split()) < 2:
        claude_response = call_claude_fallback(
            query   = beckn_intent.item,
            context = (
                "You are a procurement spec analyst. "
                "Strip highly specific technical specs (pressure ratings, material grade codes, "
                "dimensional specs, standards certifications) from the procurement item name below. "
                "Preserve the core product category and primary material family. "
                "Return ONLY the simplified item name as plain text, no explanation."
            ),
        )
        broadened_item = claude_response.splitlines()[0].strip()

    return broadened_item, kept_descriptions


print("broaden_procurement_query() defined.")

broaden_procurement_query() defined.


### 11e. Production Recovery Actions (Mocks)

These three functions represent the **side-effect layer** triggered when both the semantic cache
and the MCP fallback fail and query broadening finds no match. In production:

| Function | Production Implementation | PoC Behaviour |
|---|---|---|
| `log_unmet_demand()` | INSERT into `unmet_demand_log` PostgreSQL table | Print to stdout |
| `notify_buyer_no_stock()` | Emit event to frontend notification service | Print to stdout |
| `trigger_open_rfq_flow()` | Publish `BecknIntent` to Kafka → RFQ Lambda | Print to stdout |

All three are **fire-and-forget** in production — they must not add latency to the Lambda 1 response path.

In [18]:
def log_unmet_demand(item: str, specs: list[str]) -> None:
    """
    Production: INSERT into unmet_demand_log (item_name, descriptions, logged_at, …).
    PoC: prints the record that would be written.
    """
    print(f"  [LOG_UNMET_DEMAND] item='{item}' | specs={specs}")


def notify_buyer_no_stock(item: str) -> None:
    """
    Production: emit a notification event to the frontend service.
    PoC: prints the message that would be delivered to the buyer UI.
    """
    msg = (
        f"No supplier currently registered on the Beckn network carries an exact match "
        f"for '{item}'. Your request has been logged and an open RFQ has been broadcast."
    )
    print(f"  [NOTIFY_BUYER] {msg}")


def trigger_open_rfq_flow(beckn_intent: BecknIntent) -> None:
    """
    Production: publish BecknIntent as an open RFQ to Kafka → RFQ Publisher Lambda.
    The Beckn /search endpoint broadcasts the unmet demand to all registered BPPs.
    PoC: prints the RFQ payload that would be published.
    """
    rfq_payload = {
        "item":     beckn_intent.item,
        "specs":    beckn_intent.descriptions,
        "quantity": beckn_intent.quantity,
        "location": beckn_intent.location_coordinates,
        "budget":   beckn_intent.budget_constraints.model_dump() if beckn_intent.budget_constraints else None,
        "timeline": beckn_intent.delivery_timeline,
    }
    print(f"  [TRIGGER_RFQ] Broadcasting open RFQ to Beckn network: {rfq_payload}")


print("Recovery action mocks defined: log_unmet_demand, notify_buyer_no_stock, trigger_open_rfq_flow")

Recovery action mocks defined: log_unmet_demand, notify_buyer_no_stock, trigger_open_rfq_flow


## 12. Stage 3 — Orchestrator

In [19]:
def run_stage3_hybrid_validation(
    item_name:              str,
    extracted_descriptions: list[str],
) -> ValidationResult:
    """
    Stage 3 hybrid BPP item existence validation.

    1. Embed item_name + descriptions (richer query format).
    2. HNSW ANN search (ef_search=100).
    3. Three-zone threshold decision.
    4. CACHE_MISS only: call MCP fallback.
       - MCP found   → write Path B row → return mcp_validated=True
       - MCP not found → return not_found=True (orchestrator triggers recovery)
    """
    # Always embed with the richer format to match Path B quality
    query_embed_input = (
        item_name + " | " + " ".join(extracted_descriptions)
        if extracted_descriptions
        else item_name
    )
    query_embedding = embed(query_embed_input)

    ann_hits          = query_semantic_cache(query_embedding, top_k=3)
    zone, top_ann_hit = apply_three_zone_threshold(ann_hits)

    if zone == ValidationZone.VALIDATED:
        return ValidationResult(
            zone              = ValidationZone.VALIDATED,
            similarity        = float(top_ann_hit["cosine_similarity"]),
            matched_item_name = top_ann_hit["item_name"],
            matched_bpp_id    = top_ann_hit["bpp_id"],
        )

    if zone == ValidationZone.AMBIGUOUS:
        ambiguous_suggestions = [
            hit["item_name"]
            for hit in ann_hits
            if float(hit["cosine_similarity"]) >= AMBIGUOUS_THRESHOLD
        ]
        return ValidationResult(
            zone        = ValidationZone.AMBIGUOUS,
            similarity  = float(top_ann_hit["cosine_similarity"]),
            suggestions = ambiguous_suggestions,
        )

    # CACHE_MISS — call MCP fallback tool
    mcp_response = mock_mcp_search_bpp_catalog(item_name)

    if not mcp_response.get("found"):
        # not_found=True signals the orchestrator to run the production recovery flow
        return ValidationResult(zone=ValidationZone.CACHE_MISS, not_found=True)

    # MCP found the item — write an enriched Path B row into the cache
    write_path_b_cache_row(
        item_name    = mcp_response["item_name"],
        descriptions = mcp_response["descriptions"],
        bpp_id       = mcp_response["bpp_id"],
        bpp_uri      = mcp_response["bpp_uri"],
        provider_id  = mcp_response.get("provider_id"),
        category_tag = mcp_response.get("category_tag"),
    )

    return ValidationResult(
        zone              = ValidationZone.CACHE_MISS,
        matched_item_name = mcp_response["item_name"],
        matched_bpp_id    = mcp_response["bpp_id"],
        mcp_validated     = True,
        path_b_written    = True,
    )


print("run_stage3_hybrid_validation() defined.")

run_stage3_hybrid_validation() defined.


## 13. Cache Reset + Path A Seeding

Truncate the cache for a reproducible run, then seed 3 items via Path A (`CatalogCacheWriter`).  
This simulates the BPP `on_discover` callback that runs before any procurement query arrives.

In [20]:
# ── Reset cache for a clean, reproducible PoC run ────────────────────────────
with get_db_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("TRUNCATE TABLE bpp_catalog_semantic_cache;")
    conn.commit()
print("Cache cleared.")

# ── Seed via Path A (item_name_only embeddings) ────────────────────────────────
print("\nSeeding cache via Path A (CatalogCacheWriter — source=bpp_publish)...")
seed_items = mock_catalog_normalizer_path_a()

for catalog_item in seed_items:
    print(f"  → '{catalog_item['item_name']}' [{catalog_item['category_tag']}]")
    write_path_a_cache_row(
        item_name    = catalog_item["item_name"],
        bpp_id       = catalog_item["bpp_id"],
        bpp_uri      = catalog_item["bpp_uri"],
        provider_id  = catalog_item.get("provider_id"),
        category_tag = catalog_item.get("category_tag"),
    )

with get_db_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM bpp_catalog_semantic_cache;")
        seeded_row_count = cur.fetchone()[0]

print(f"\nCache seeded. Total rows: {seeded_row_count}")
assert seeded_row_count == len(seed_items), "Seeding produced unexpected row count"

Cache cleared.

Seeding cache via Path A (CatalogCacheWriter — source=bpp_publish)...
  → 'Stainless Steel Flanged Ball Valve 2 inch' [valves]
  → 'Industrial HDPE Pipe 6 inch' [pipes]
  → 'Safety Helmet Class E Hard Hat' [ppe]

Cache seeded. Total rows: 3


## 14. Unified Pipeline — `parse_procurement_request()`

This is the master orchestrator that chains all 3 stages.  
A non-procurement intent short-circuits after Stage 1 — Stages 2 and 3 are never invoked.

In [21]:
# Procurement intents that gate Stage 2 & 3 execution
_PROCUREMENT_INTENTS: frozenset[str] = frozenset({
    "SearchProduct",
    "RequestQuote",
    "PurchaseOrder",
})

# Module-level singletons (one instance per notebook session)
_intent_classifier = IntentClassifier()
_beckn_parser      = BecknIntentParser()


def parse_procurement_request(raw_query: str) -> dict:
    """
    Master E2E orchestrator — Stage 1 → Stage 2 → Stage 3 → Recovery (if needed).

    Stage 1 : classify intent → gate non-procurement
    Stage 2 : extract BecknIntent (item, descriptions, location, budget…)
    Stage 3 : hybrid BPP item validation (cache → MCP)
    Recovery: triggered only on CACHE_MISS + MCP complete failure (not_found=True)
      Step 1 — broaden_procurement_query   (heuristic + Claude if degenerate)
      Step 2 — retry Stage 3 with broader query
      Step 3 — if still NOT_FOUND: log_unmet_demand + notify_buyer + trigger_rfq
      Step 4 — call_claude_fallback as Senior Auditor for a final assessment

    Returns ParseResponse dict:
      intent        (str)              — classified intent label
      confidence    (float)            — Stage 1 confidence
      beckn_intent  (dict | None)      — Stage 2 extraction (model_dump)
      routed_to     (str | None)       — model used by Stage 2
      validation    (ValidationResult) — Stage 3 outcome (possibly broadened)
      recovery_log  (list[str])        — actions taken in recovery (empty if not triggered)
    """
    recovery_log: list[str] = []

    # ── Stage 1: Intent Classification ───────────────────────────────────────
    stage1_parsed_intent: ParsedIntent = _intent_classifier.classify(raw_query)

    if stage1_parsed_intent.intent not in _PROCUREMENT_INTENTS:
        return {
            "intent":       stage1_parsed_intent.intent,
            "confidence":   stage1_parsed_intent.confidence,
            "beckn_intent": None,
            "routed_to":    None,
            "validation":   None,
            "recovery_log": recovery_log,
        }

    # ── Stage 2: BecknIntent Extraction ──────────────────────────────────────
    extracted_beckn_intent: BecknIntent = _beckn_parser.parse(raw_query)
    stage2_model_used:      str         = _beckn_parser.last_model_used

    # ── Stage 3: Hybrid BPP Item Existence Validation ─────────────────────────
    validation_result: ValidationResult = run_stage3_hybrid_validation(
        item_name              = extracted_beckn_intent.item,
        extracted_descriptions = extracted_beckn_intent.descriptions,
    )

    # ── Production Recovery: CACHE_MISS + complete MCP failure ────────────────
    # Triggered only when not_found=True (both cache and MCP returned nothing).
    # AMBIGUOUS and MCP_VALIDATED paths are NOT recovery scenarios.
    if validation_result.zone == ValidationZone.CACHE_MISS and validation_result.not_found:

        # Step 1 — Query Broadening: strip technical specs, retry at category level
        broadened_item_name, broadened_descriptions = broaden_procurement_query(
            extracted_beckn_intent
        )
        recovery_log.append(f"[BROADEN] '{extracted_beckn_intent.item}' → '{broadened_item_name}'")

        # Step 2 — Retry Stage 3 with the broader item + kept descriptions
        broad_validation: ValidationResult = run_stage3_hybrid_validation(
            item_name              = broadened_item_name,
            extracted_descriptions = broadened_descriptions,
        )
        broad_validation.broadened_item_name = broadened_item_name

        if broad_validation.zone in (ValidationZone.VALIDATED, ValidationZone.AMBIGUOUS):
            # Broadening found a partial match — surface it with the broadened label
            sim_str = (
                f"  similarity={broad_validation.similarity:.4f}"
                if broad_validation.similarity is not None
                else ""
            )
            recovery_log.append(f"[BROAD_HIT] zone={broad_validation.zone.value}{sim_str}")
            validation_result = broad_validation

        else:
            # Step 3 — Broadening also failed: invoke all three recovery actions
            log_unmet_demand(
                item  = extracted_beckn_intent.item,
                specs = extracted_beckn_intent.descriptions,
            )
            notify_buyer_no_stock(item=extracted_beckn_intent.item)
            trigger_open_rfq_flow(beckn_intent=extracted_beckn_intent)
            recovery_log.append("[RECOVERY] log_unmet_demand + notify_buyer_no_stock + trigger_open_rfq_flow")

            # Step 4 — Claude Senior Auditor: final assessment (cloud, last resort)
            claude_audit: str = call_claude_fallback(
                query   = extracted_beckn_intent.item,
                context = (
                    "You are a senior procurement auditor. "
                    "The local pipeline returned no results for this item and query broadening also failed. "
                    "Assess in one sentence: is this a niche/specialty item, a genuine Beckn network supply gap, "
                    "or a likely LLM extraction error?"
                ),
            )
            recovery_log.append(f"[CLAUDE_AUDIT] {claude_audit}")
            validation_result.not_found = True

    return {
        "intent":       stage1_parsed_intent.intent,
        "confidence":   stage1_parsed_intent.confidence,
        "beckn_intent": extracted_beckn_intent.model_dump(),
        "routed_to":    stage2_model_used,
        "validation":   validation_result,
        "recovery_log": recovery_log,
    }


print("parse_procurement_request() defined — Day-2 recovery flow active.")
print(f"Procurement gate: {sorted(_PROCUREMENT_INTENTS)}")

parse_procurement_request() defined — Day-2 recovery flow active.
Procurement gate: ['PurchaseOrder', 'RequestQuote', 'SearchProduct']


## 15. Display Helper

In [22]:
def print_parse_response(label: str, raw_query: str, response: dict) -> None:
    """Pretty-print a ParseResponse dict for E2E test output."""
    bar = "=" * 68
    print(bar)
    print(f"  {label}")
    print(bar)
    print(f"  Query      : {raw_query}")
    print(f"  Stage 1    : intent={response['intent']}  confidence={response['confidence']:.0%}")

    extracted_beckn_intent = response.get("beckn_intent")
    if extracted_beckn_intent is None:
        print("  Stage 2&3  : SKIPPED — non-procurement intent")
        print(bar)
        return

    print(f"  Stage 2    : item='{extracted_beckn_intent['item']}'")
    print(f"               descriptions={extracted_beckn_intent['descriptions']}")
    print(
        f"               quantity={extracted_beckn_intent['quantity']}  "
        f"location={extracted_beckn_intent['location_coordinates']}"
    )
    print(
        f"               timeline={extracted_beckn_intent['delivery_timeline']}h  "
        f"budget=[{extracted_beckn_intent['budget_constraints']['min']}, "
        f"{extracted_beckn_intent['budget_constraints']['max']}]"
    )
    print(f"               routed_to={response['routed_to']}")

    stage3_validation: ValidationResult = response["validation"]
    print(f"  Stage 3    : {stage3_validation}")

    if stage3_validation.broadened_item_name:
        print(f"  Broadened  : original spec-item → '{stage3_validation.broadened_item_name}'")

    if stage3_validation.zone == ValidationZone.AMBIGUOUS and stage3_validation.suggestions:
        print("  Suggestions for user confirmation:")
        for suggestion in stage3_validation.suggestions:
            print(f"               · {suggestion}")

    recovery_log: list[str] = response.get("recovery_log", [])
    if recovery_log:
        print("  Recovery   :")
        for entry in recovery_log:
            print(f"               {entry}")

    print(bar)


print("print_parse_response() defined.")

print_parse_response() defined.


---

## 16. End-to-End Test Cases

Each test passes a **raw natural language query** through the full `parse_procurement_request()` pipeline.

| Test | Raw NL Query | Expected Stage 3 Outcome |
|---|---|---|
| TC1 | Stainless steel flanged ball valves (specific specs) | VALIDATED — direct cache hit (≥ 0.85) |
| TC2 | Industrial pipeline components, fluid transport | AMBIGUOUS — suggestions gate (0.45–0.84) |
| TC3 | Cat6 UTP network cable, 300m, Mumbai | CACHE_MISS → MCP → Path B cache write |
| Gate | Good morning, office hours? | Non-procurement — Stage 2 & 3 skipped |
| **TC4** | Super Duplex ASTM A790 pipe fitting, DN100 PN40, offshore | **CACHE_MISS → MCP FAIL → Broadening → Recovery** |

In [23]:
tc1_raw_query = (
    "I need 50 units of stainless steel flanged ball valves, 2-inch PN16 rated, "
    "for our Bengaluru refinery. Budget 4500 INR per unit, delivery within 5 days."
)

tc1_response = parse_procurement_request(tc1_raw_query)
print_parse_response("TC1 — Expected: VALIDATED", tc1_raw_query, tc1_response)

# Assertions
tc1_validation: ValidationResult = tc1_response["validation"]
assert tc1_response["intent"] in _PROCUREMENT_INTENTS, \
    f"TC1: Stage 1 gate failed — got intent='{tc1_response['intent']}'"
assert tc1_response["beckn_intent"] is not None, \
    "TC1: Stage 2 produced no BecknIntent"

if tc1_validation.zone == ValidationZone.VALIDATED:
    print(f"\n✓ PASS — VALIDATED  similarity={tc1_validation.similarity:.4f}")
    print(f"  Matched BPP item : '{tc1_validation.matched_item_name}'")
else:
    print(f"\n⚠ Zone was {tc1_validation.zone.value} (similarity={tc1_validation.similarity})")
    print("  Note: LLM extracted a different item_name — adjust query or check Ollama output.")
    print(f"  Extracted item   : '{tc1_response['beckn_intent']['item']}'")

  TC1 — Expected: VALIDATED
  Query      : I need 50 units of stainless steel flanged ball valves, 2-inch PN16 rated, for our Bengaluru refinery. Budget 4500 INR per unit, delivery within 5 days.
  Stage 1    : intent=RequestQuote  confidence=95%
  Stage 2    : item='stainless steel flanged ball valves'
               descriptions=['2-inch', 'PN16']
               quantity=50  location=12.9716,77.5946
               timeline=120h  budget=[0.0, 4500.0]
               routed_to=qwen3:8b
  Stage 3    : ValidationResult(zone=VALIDATED, similarity=0.8730, matched='Stainless Steel Flanged Ball Valve 2 inch')

✓ PASS — VALIDATED  similarity=0.8730
  Matched BPP item : 'Stainless Steel Flanged Ball Valve 2 inch'


---

### Test Case 2 — AMBIGUOUS (Suggestions Gate)

A generic industrial query. Stage 2 should extract a broad item name semantically related  
to cached items (pipeline / pipe fittings) but not an exact match, landing in the 0.75–0.91 range.

In [24]:
tc2_raw_query = (
    "Our Chennai plant needs industrial pipeline components or pipe fittings, "
    "something suitable for heavy-duty fluid transport systems."
)

tc2_response = parse_procurement_request(tc2_raw_query)
print_parse_response("TC2 — Expected: AMBIGUOUS", tc2_raw_query, tc2_response)

tc2_validation: ValidationResult = tc2_response["validation"]
assert tc2_response["beckn_intent"] is not None, \
    "TC2: Stage 2 produced no BecknIntent"

print(f"\nExtracted item : '{tc2_response['beckn_intent']['item']}'")
print(f"Similarity     : {tc2_validation.similarity}")

if tc2_validation.zone == ValidationZone.AMBIGUOUS:
    print(f"\n✓ PASS — AMBIGUOUS  similarity={tc2_validation.similarity:.4f}")
    print("  User confirmation gate triggered. Suggestions:")
    for s in tc2_validation.suggestions:
        print(f"    · {s}")
elif tc2_validation.zone == ValidationZone.VALIDATED:
    print(f"\n⚠ VALIDATED (similarity={tc2_validation.similarity:.4f}) — stronger match than expected.")
    print(f"  Extracted item '{tc2_response['beckn_intent']['item']}' matched closely.")
else:
    print(f"\n⚠ CACHE_MISS — item fell below AMBIGUOUS threshold.")
    print(f"  Extracted item was too far from cached items.")

  TC2 — Expected: AMBIGUOUS
  Query      : Our Chennai plant needs industrial pipeline components or pipe fittings, something suitable for heavy-duty fluid transport systems.
  Stage 1    : intent=SearchProduct  confidence=95%
  Stage 2    : item='industrial pipeline components or pipe fittings'
               descriptions=['heavy-duty']
               quantity=0  location=13.0827,80.2707
               timeline=1h  budget=[0.0, 0.0]
               routed_to=qwen3:8b
  Stage 3    : ValidationResult(zone=AMBIGUOUS, similarity=0.5672, suggestions=['Industrial HDPE Pipe 6 inch'])
  Suggestions for user confirmation:
               · Industrial HDPE Pipe 6 inch

Extracted item : 'industrial pipeline components or pipe fittings'
Similarity     : 0.5672096787912237

✓ PASS — AMBIGUOUS  similarity=0.5672
  User confirmation gate triggered. Suggestions:
    · Industrial HDPE Pipe 6 inch


---

### Test Case 3 — CACHE MISS → MCP Fallback → Path B Write

A networking cable query. Stage 2 extracts an item completely outside the seeded catalog  
(valves / pipes / helmets). Stage 3 gets a CACHE_MISS, calls the MCP mock, finds "UTP Cat6 Network Cable",  
and writes a new enriched Path B row into the DB.

In [25]:
tc3_raw_query = (
    "We need to procure 300 meters of Cat6 UTP network cable for our new "
    "office building in Mumbai. Delivery in 5 days, budget under 20 INR per meter."
)

# Snapshot Path B count before the test
with get_db_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT COUNT(*) FROM bpp_catalog_semantic_cache WHERE source = 'mcp_feedback';"
        )
        path_b_count_before: int = cur.fetchone()[0]

tc3_response = parse_procurement_request(tc3_raw_query)
print_parse_response("TC3 — Expected: CACHE MISS → MCP → Path B", tc3_raw_query, tc3_response)

# Snapshot Path B count after
with get_db_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT COUNT(*) FROM bpp_catalog_semantic_cache WHERE source = 'mcp_feedback';"
        )
        path_b_count_after: int = cur.fetchone()[0]

tc3_validation: ValidationResult = tc3_response["validation"]

print(f"\nExtracted item      : '{tc3_response['beckn_intent']['item']}'")
print(f"Path B rows before  : {path_b_count_before}")
print(f"Path B rows after   : {path_b_count_after}")

if (
    tc3_validation.zone == ValidationZone.CACHE_MISS
    and tc3_validation.mcp_validated
    and tc3_validation.path_b_written
    and path_b_count_after == path_b_count_before + 1
):
    print(f"\n✓ PASS — CACHE_MISS → MCP_VALIDATED → Path B written")
    print(f"  MCP matched item  : '{tc3_validation.matched_item_name}'")
    print(f"  BPP               : {tc3_validation.matched_bpp_id}")
elif tc3_validation.zone == ValidationZone.CACHE_MISS and not tc3_validation.mcp_validated:
    print(f"\n⚠ CACHE_MISS — MCP mock did not match.")
    print(f"  Extracted item '{tc3_response['beckn_intent']['item']}' "
          "did not contain a known MCP key ('cable').")
    print("  Consider adjusting the TC3 query or expanding _MCP_MOCK_RESPONSES.")
else:
    print(f"\n⚠ Unexpected zone: {tc3_validation.zone.value}")

  TC3 — Expected: CACHE MISS → MCP → Path B
  Query      : We need to procure 300 meters of Cat6 UTP network cable for our new office building in Mumbai. Delivery in 5 days, budget under 20 INR per meter.
  Stage 1    : intent=RequestQuote  confidence=95%
  Stage 2    : item='Cat6 UTP network cable'
               descriptions=['Cat6']
               quantity=300  location=19.0760,72.8777
               timeline=120h  budget=[0.0, 6000.0]
               routed_to=qwen3:8b
  Stage 3    : ValidationResult(zone=CACHE_MISS, matched='UTP Cat6 Network Cable', mcp_validated=True, path_b_written=True)

Extracted item      : 'Cat6 UTP network cable'
Path B rows before  : 0
Path B rows after   : 1

✓ PASS — CACHE_MISS → MCP_VALIDATED → Path B written
  MCP matched item  : 'UTP Cat6 Network Cable'
  BPP               : bpp_it_supplies_003


---

### Bonus: Non-Procurement Intent Gate (Stage 1 Short-Circuit)

In [26]:
gate_raw_query = "Good morning, can you tell me your office hours?"

gate_response = parse_procurement_request(gate_raw_query)
print_parse_response("GATE — Non-procurement (Stage 2 & 3 should be skipped)", gate_raw_query, gate_response)

assert gate_response["beckn_intent"] is None, "Gate failed: Stage 2 ran on a non-procurement intent"
assert gate_response["validation"]   is None, "Gate failed: Stage 3 ran on a non-procurement intent"
print(f"✓ PASS — intent='{gate_response['intent']}' correctly gated out.")

  GATE — Non-procurement (Stage 2 & 3 should be skipped)
  Query      : Good morning, can you tell me your office hours?
  Stage 1    : intent=OfficeHoursInquiry  confidence=50%
  Stage 2&3  : SKIPPED — non-procurement intent
✓ PASS — intent='OfficeHoursInquiry' correctly gated out.


---

### Test Case 4 — CACHE_MISS → MCP FAIL → Query Broadening → Recovery Actions

A highly specialized offshore procurement item with multiple technical constraints:
ASTM grade, UNS designation, nominal bore rating, and pressure class. None of these
appear in the seeded cache or the MCP mock responses, triggering the full recovery path.

**Expected flow:**
```
Stage 3 initial : CACHE_MISS (no semantic match in cache)
MCP probe       : NOT_FOUND  (no MCP key matches 'pipe fitting' or 'duplex')
not_found=True  → recovery triggered
  [BROADEN]     : strip ASTM, S32750, DN100, PN40 → category-level item name
  Broad Stage 3 : CACHE_MISS or AMBIGUOUS (depends on broadened result vs. cache)
  [RECOVERY]    : log_unmet_demand + notify_buyer + trigger_open_rfq (if still not found)
  [CLAUDE_AUDIT]: Senior Auditor assessment (placeholder if key not set)
```

In [27]:
tc4_raw_query = (
    "We require 20 units of ASTM A790 Super Duplex S32750 stainless steel "
    "butt-weld pipe fittings, DN100 PN40 rated, for our offshore oil platform "
    "project at Mumbai port. Urgency 72 hours. Budget INR 45,000 per unit."
)

tc4_response = parse_procurement_request(tc4_raw_query)
print_parse_response("TC4 — Expected: CACHE_MISS → MCP FAIL → Recovery", tc4_raw_query, tc4_response)

tc4_validation: ValidationResult = tc4_response["validation"]
tc4_recovery:   list[str]        = tc4_response["recovery_log"]

print(f"\nExtracted item  : '{tc4_response['beckn_intent']['item']}'")
print(f"Recovery steps  : {len(tc4_recovery)}")
for entry in tc4_recovery:
    print(f"  {entry}")

# ── Assertions ────────────────────────────────────────────────────────────
assert tc4_recovery, "TC4: recovery_log must be non-empty — the full failure path was not reached"
assert any("[BROADEN]" in e for e in tc4_recovery), "TC4: Query Broadening must be attempted"

if any("[BROAD_HIT]" in e for e in tc4_recovery):
    # Broadening found a partial match — valid outcome, recovery partially succeeded
    print(f"\nRESULT: Query Broadening found a partial match → zone={tc4_validation.zone.value}")
    print(f"        Broadened item: '{tc4_validation.broadened_item_name}'")
    if tc4_validation.suggestions:
        print("        Suggestions returned to buyer:")
        for s in tc4_validation.suggestions:
            print(f"          · {s}")
    print("✓ PASS — Partial recovery via Query Broadening")

elif any("[RECOVERY]" in e for e in tc4_recovery):
    # Full recovery actions triggered — complete dead-end scenario
    print("\nRESULT: Complete supply gap — all recovery actions invoked")
    print(f"        Final zone : {tc4_validation.zone.value}")
    claude_entries = [e for e in tc4_recovery if "[CLAUDE_AUDIT]" in e]
    if claude_entries:
        print(f"        Claude audit: {claude_entries[0]}")
    elif not CLAUDE_FALLBACK_ENABLED:
        print("        (Claude fallback disabled — no ANTHROPIC_API_KEY in environment)")
    print("✓ PASS — Full recovery path executed (log + notify + RFQ + Claude audit)")

else:
    print(f"\n⚠ Unexpected recovery path. Log: {tc4_recovery}")

  TC4 — Expected: CACHE_MISS → MCP FAIL → Recovery
  Query      : We require 20 units of ASTM A790 Super Duplex S32750 stainless steel butt-weld pipe fittings, DN100 PN40 rated, for our offshore oil platform project at Mumbai port. Urgency 72 hours. Budget INR 45,000 per unit.
  Stage 1    : intent=RequestQuote  confidence=95%
  Stage 2    : item='ASTM A790 Super Duplex S32750 stainless steel butt-weld pipe fittings'
               descriptions=['DN100', 'PN40', 'butt-weld', 'ASTM A790 Super Duplex S32750']
               quantity=20  location=19.0760,72.8777
               timeline=72h  budget=[0.0, 900000.0]
               routed_to=qwen3:8b
  Stage 3    : ValidationResult(zone=AMBIGUOUS, similarity=0.4605, suggestions=['Industrial HDPE Pipe 6 inch'], broadened='Super Duplex stainless steel butt-weld pipe fittings')
  Broadened  : original spec-item → 'Super Duplex stainless steel butt-weld pipe fittings'
  Suggestions for user confirmation:
               · Industrial HDPE Pipe 6 in

---

## 17. Final Cache State

In [28]:
print("=" * 68)
print("  FINAL CACHE STATE")
print("=" * 68)

with get_db_connection() as conn:
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute("""
            SELECT
                item_name,
                bpp_id,
                source,
                embedding_strategy,
                hit_count,
                CASE
                    WHEN descriptions IS NULL THEN 'NULL'
                    ELSE array_to_string(descriptions, ', ')
                END AS descriptions_display
            FROM bpp_catalog_semantic_cache
            ORDER BY created_at;
        """)
        final_cache_rows = cur.fetchall()

header = f"{'#':<3} {'item_name':<44} {'source':<14} {'strategy':<22} {'hits':<5} descriptions"
print(header)
print("-" * 115)
for row_num, row in enumerate(final_cache_rows, 1):
    source_display   = row["source"]
    strategy_display = row["embedding_strategy"]
    print(
        f"{row_num:<3} "
        f"{row['item_name'][:43]:<44} "
        f"{source_display:<14} "
        f"{strategy_display:<22} "
        f"{row['hit_count']:<5} "
        f"{row['descriptions_display']}"
    )

path_a_rows = sum(1 for r in final_cache_rows if r["source"] == "bpp_publish")
path_b_rows = sum(1 for r in final_cache_rows if r["source"] == "mcp_feedback")
print(f"\nTotal: {len(final_cache_rows)} rows  |  Path A (bpp_publish): {path_a_rows}  |  Path B (mcp_feedback): {path_b_rows}")

  FINAL CACHE STATE
#   item_name                                    source         strategy               hits  descriptions
-------------------------------------------------------------------------------------------------------------------
1   Stainless Steel Flanged Ball Valve 2 inch    bpp_publish    item_name_only         0     NULL
2   Industrial HDPE Pipe 6 inch                  bpp_publish    item_name_only         0     NULL
3   Safety Helmet Class E Hard Hat               bpp_publish    item_name_only         0     NULL
4   UTP Cat6 Network Cable                       mcp_feedback   item_name_and_specs    0     Cat6, UTP, unshielded twisted pair, 550MHz, 23AWG

Total: 4 rows  |  Path A (bpp_publish): 3  |  Path B (mcp_feedback): 1


---

## Summary

| Stage | Component | Tech | Role |
|---|---|---|---|
| 1 | `IntentClassifier` | Ollama `qwen3:8b` + `instructor` | Classify intent, gate non-procurement |
| 2 | `BecknIntentParser` | Ollama `qwen3:8b/1.7b` + `instructor` | Extract `BecknIntent` (item, specs, location, budget) |
| 3 | `run_stage3_hybrid_validation` | `all-MiniLM-L6-v2` (384d, local CPU) + pgvector HNSW | Validate item existence in BPP semantic cache |
| Recovery | `broaden_procurement_query` + recovery mocks | Heuristic regex + Claude (last resort) | Query Broadening → Logging → Notification → Open RFQ |

| Test Case | Input | Stage 1 | Stage 2 Item | Stage 3 Zone |
|---|---|---|---|---|
| TC1 | Stainless steel flanged ball valves, PN16 | RequestQuote | SS flanged ball valve | **VALIDATED** ≥ 0.85 |
| TC2 | Industrial pipeline components, fluid transport | SearchProduct | pipeline components | **AMBIGUOUS** 0.45–0.84 |
| TC3 | Cat6 UTP network cable, 300m, Mumbai | RequestQuote | Cat6 UTP cable | **CACHE_MISS** → MCP → Path B |
| Gate | Good morning, can you tell me… | GeneralInquiry | — | — (short-circuited) |
| **TC4** | Super Duplex ASTM A790 pipe fitting, offshore | RequestQuote | SS butt-weld pipe fitting | **CACHE_MISS → MCP FAIL → Broadening → Recovery** |

**Key architecture decisions preserved:**
- **Local-First principle**: Stages 1, 2, and 3 are 100% local (Ollama + sentence-transformers + pgvector). Cloud calls (Claude) are gated behind `CLAUDE_FALLBACK_ENABLED` and invoked only on confirmed failure paths.
- **Recovery flow is orchestrator-owned**: `run_stage3_hybrid_validation()` stays stateless and single-responsibility. Recovery logic lives exclusively in `parse_procurement_request()`.
- **`not_found=True` is the recovery signal**: A clean contract between the Stage 3 validator and the orchestrator — no implicit side effects in the validation layer.
- **Query Broadening is heuristic-first**: Regex strips standards codes, grade codes, and dimensional specs. Claude is invoked only if the heuristic produces a degenerate result (< 2 tokens).
- **`write_path_b_cache_row()` upgrades Path A rows in-place via UPSERT** — the cache self-improves on successful MCP hits.